# Block 2: Volume Ingestion

---

## Agenda
| Section | Duration |
|---|---|
| Guided: Connecting to Volumes | 30 min |
| Guided: Auto Loader for Incremental Processing | 30 min |
| Hands-on Exercise | 15 min |

---
## Guided Section: Connecting to Volumes (30 min)

In [0]:
%run ./_resources/Config

## 1. Introduction to Volumes



## 2. Batch Read from Volumes


In [0]:
# ============================================================
# Batch read binary files from Volumes
# ============================================================
from pyspark.sql.functions import regexp_extract, substring

df = (spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", True)
    .load(f"/Volumes/{catalog}/{shared_schema}/{test_documents_volume}")
    .withColumn("fileName", regexp_extract("path", r"([^/]+)$", 1))
)

display(df.select("fileName", "path", "modificationTime", "length", substring("content", 1, 100).alias("content_preview")))

## 3. File Type Filtering with `pathGlobFilter`

The `pathGlobFilter` option lets you control which file types are ingested at the source — before any data is loaded into memory.

**Common patterns:**

| Pattern | What it matches |
|---|---|
| `*.pdf` | Only PDF files |
| `*.pptx` | Only PowerPoint files |
| `*.docx` | Only Word documents |
| `*.{pdf,pptx,docx}` | PDFs, PowerPoints, and Word docs |
| `*.{pdf,pptx,docx,xlsx}` | Add Excel spreadsheets too |


In [0]:
# ============================================================
# Batch read binary files
# ============================================================

df = (spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pdf")
    .load(f"/Volumes/{catalog}/{shared_schema}/{test_documents_volume}")
    .withColumn("fileName", regexp_extract("path", r"([^/]+)$", 1))
)

display(df.select("fileName", "path", "modificationTime", "length", substring("content", 1, 100).alias("content_preview")))

## 4. Metadata

Volumes expose a special `_metadata` struct column that contains rich metadata.


This metadata lets you filter, tag, and route documents.


In [0]:
# ============================================================
# Explore metadata
# ============================================================

df_meta = (spark.read
    .format("binaryFile")
    .load(f"/Volumes/{catalog}/{shared_schema}/{test_documents_volume}")
    .select("path", "length", "_metadata")
)

display(df_meta)

In [0]:
# Filter by metadata — e.g., only PDF files based on MIME type
df_meta.filter("_metadata.file_size > 100000").display()

---
## Guided Section: Auto Loader for Incremental Processing (30 min)

## 5. Auto Loader Introduction

**Auto Loader** (`cloudFiles` format) is Databricks' built-in solution for incrementally processing new files as they arrive. When combined with Volumes, it replaces the need for:

- **EventBridge rules** that trigger on a schedule
- **Lambda functions** that poll the Graph API for changes
- **Custom DynamoDB tables** that track which files have been processed

All of that is handled automatically by Auto Loader's checkpoint mechanism.

![Lakeflow Connect Architecture](./images/lakeflow_connect_unified.png)

**How it works:**

| Run | Behavior | What gets processed |
|---|---|---|
| First run | **Full crawl** (`includeExistingFiles=True`) | ALL existing files in the library |
| Subsequent runs | **Incremental** | Only new or modified files since last checkpoint |


## 6. Streaming Read with Auto Loader

The code below sets up a streaming pipeline that:
1. Reads all existing files on the first run (full crawl)
2. On subsequent runs, processes only new/modified files (incremental)
3. Writes everything to a Delta table (`01_bronze_raw_documents`)
4. Uses `trigger(availableNow=True)` to process all available files and then stop

> `availableNow=True` processes everything available right now and stops, which is ideal for scheduled jobs. 

>For real-time ingestion, you would use something like `trigger(processingTime="5 minutes")` instead.

### Detect page count per PDF

Knowing the number of pages in PDF documents will be important when parsing documents in the next module.

The notebook `./_resources/Utilities` defines a Pandas User Defined Function (UDF) that uses the `pypdf` library to determine the page count for PDF documents. 
- Bad/encrypted files return `-1` so they surface in downstream filters without killing the job
- Non-PDF documents also return `-1`

In [0]:
%run ./_resources/Utilities

In [0]:
# ============================================================
# Auto Loader: streaming read
# ============================================================

df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("cloudFiles.includeExistingFiles", True)
    .option("pathGlobFilter", "*.{pdf,pptx,docx}")
    .load(f"/Volumes/{catalog}/{shared_schema}/{test_documents_volume}")
    .withColumn("fileName", regexp_extract("path", r"([^/]+)$", 1))
    .withColumn("page_count", pdf_page_count("content"))
)

# Write to a Delta table with checkpoint tracking
query = (df_stream.writeStream
    .option("checkpointLocation", f"/Volumes/{catalog}/{schema}/{personal_volume}/checkpoints/raw_docs_volume")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{schema}.01_bronze_raw_documents"))

In [0]:
display(spark.read.table(f"{catalog}.{schema}.01_bronze_raw_documents")
        .select("fileName", "path", "modificationTime", "length", substring("content", 1, 100)))

### Incremental ingestion
1.  Drag a new file (pdf, pptx, or doc) into the Volume
1.  Rerun the cell `Autoload files`
1.  Rerun the cell `Examine the Bronze table`.  Note there is one additional row.

## 7. Full vs. Incremental Crawl

Understanding how Auto Loader manages state is critical for building reliable pipelines.

**First run (Full Crawl):**
- `includeExistingFiles=True` tells Auto Loader to process **all existing files** in the library
- Every file is read, converted to a row, and written to the Delta table
- A checkpoint is saved recording what has been processed

**Subsequent runs (Incremental):**
- Auto Loader reads the checkpoint to determine what has already been processed
- Only **new or modified files** since the last run are picked up
- The checkpoint is updated after each successful run

**Key points:**
- The **checkpoint location** (`/Volumes/{catalog}/{schema}/{volume}/checkpoints/raw_docs`) is where Auto Loader stores its state
- If you delete the checkpoint, the next run becomes a full crawl again
- You do **not** need to build custom tracking logic — Auto Loader handles it

> How do you currently track which files have been processed? Most teams have a DynamoDB table or a metadata database for this. The checkpoint replaces all of that.

---
## Recap

In this block you learned:

1. **Volumes** — read files directly from Volumes using `binaryFile` format with a Unity Catalog connection
2. **File filtering** — use `pathGlobFilter` to select specific file types at the source
3. **Metadata** — access file metadata via `_metadata` (Runtime 18+)
4. **Auto Loader** — incrementally process new files without custom tracking infrastructure
5. **Full vs. Incremental** — first run does a full crawl; subsequent runs only pick up changes

**Next up:** Block 3 — Document Parsing & Chunking